In [0]:
# =============================================================
# audit_config.py
# Layer   : Audit
# Purpose : Central metadata for standalone validation/audit runs.
#
# This file is intentionally read-only configuration. It does not
# write to Bronze, Silver, Gold, or existing pipeline notebooks.
# =============================================================

from pyspark.sql import functions as F


FACT_TABLES = [
    "sales",
    "device_snapshots",
    "activity_snapshots",
    "health_snapshots",
    "feeding_events",
    "hydration_events",
    "device_failures",
    "firmware_deployments",
]


PRIMARY_KEYS = {
    "sales": ["sale_id"],
    "device_snapshots": ["device_id", "snapshot_timestamp"],
    "activity_snapshots": ["device_id", "snapshot_timestamp"],
    "health_snapshots": ["device_id", "snapshot_timestamp"],
    "feeding_events": ["feed_event_id"],
    "hydration_events": ["water_event_id"],
    "device_failures": ["failure_id"],
    "firmware_deployments": ["deployment_id"],
}


SILVER_DATE_COLUMNS = {
    "sales": "sale_date",
    "device_snapshots": "snapshot_timestamp",
    "activity_snapshots": "snapshot_timestamp",
    "health_snapshots": "snapshot_timestamp",
    "feeding_events": "event_timestamp",
    "hydration_events": "event_timestamp",
    "device_failures": "failure_timestamp",
    "firmware_deployments": "deployment_timestamp",
}


EXPECTED_SILVER_TYPES = {
    "sales": {
        "sale_date": "date",
        "sale_price": "double",
        "discount_amount": "double",
    },
    "device_snapshots": {
        "snapshot_timestamp": "timestamp",
        "battery_pct": "double",
        "signal_strength": "int",
        "temperature_c": "double",
    },
    "activity_snapshots": {
        "snapshot_timestamp": "timestamp",
        "steps": "int",
        "distance_km": "double",
        "active_minutes": "int",
    },
    "health_snapshots": {
        "snapshot_timestamp": "timestamp",
        "heart_rate": "double",
        "pulse_rate": "double",
        "body_temperature": "double",
    },
    "feeding_events": {
        "event_timestamp": "timestamp",
        "food_dispensed_grams": "double",
    },
    "hydration_events": {
        "event_timestamp": "timestamp",
        "water_consumed_ml": "double",
    },
    "device_failures": {
        "failure_timestamp": "timestamp",
        "downtime_minutes": "int",
        "severity_score": "int",
    },
    "firmware_deployments": {
        "deployment_timestamp": "timestamp",
        "deployment_duration_seconds": "int",
        "rollback_flag": "boolean",
    },
}


def abfss_uri(container: str, path: str = "", storage_account: str = "petiot") -> str:
    clean_path = path.strip("/")
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"


def bronze_path(table_name: str, storage_account: str = "petiot") -> str:
    return abfss_uri("bronze", table_name, storage_account)


def silver_path(table_name: str, storage_account: str = "petiot") -> str:
    return abfss_uri("silver", table_name, storage_account)


def audit_path(relative_path: str, storage_account: str = "petiot", audit_container: str = "audit") -> str:
    return abfss_uri(audit_container, relative_path, storage_account)


def normalize_date(load_date: str) -> str:
    if not load_date:
        return load_date
    if load_date.startswith("year="):
        parts = load_date.replace("year=", "").replace("month=", "").replace("day=", "").split("/")
        if len(parts) == 3:
            return f"{parts[0]}-{parts[1]}-{parts[2]}"
    return load_date[:10]


def normalize_bronze_load_date(load_date: str) -> str:
    clean_date = normalize_date(load_date)
    if not clean_date:
        return clean_date
    parts = clean_date.split("-")
    if len(parts) == 3:
        return f"year={parts[0]}/month={parts[1]}/day={parts[2]}"
    return load_date


def filter_bronze_to_load_date(df, load_date: str):
    if "_load_date" not in df.columns:
        return df
    return df.filter(F.col("_load_date") == normalize_bronze_load_date(load_date))


def filter_silver_to_load_date(df, table_name: str, load_date: str):
    date_col = SILVER_DATE_COLUMNS.get(table_name)
    if not date_col or date_col not in df.columns:
        return df
    return df.filter(F.to_date(F.col(date_col)) == F.lit(normalize_date(load_date)))


def select_fact_tables(table_name: str) -> list[str]:
    clean = (table_name or "ALL_FACTS").strip()
    if clean.upper() in {"ALL", "ALL_FACTS", "*"}:
        return FACT_TABLES
    if clean not in FACT_TABLES:
        raise ValueError(f"'{clean}' is not an audited fact table. Valid: {FACT_TABLES}")
    return [clean]


print("[audit_config] Loaded fact-only audit metadata.")